This notebook will guide you through building a Tic-Tac-Toe game where an AI agent learns to play using Q-learning, a type of reinforcement learning.

We will cover the following tasks:

a. Setting up the environment: Importing necessary libraries and defining game constants.
b. Defining the Tic-Tac-Toe game: Implementing the game logic, including board representation, moves, and win conditions.
c. Building the reinforcement learning model: Creating an agent that learns through interactions with the environment.
d. Training the model: Simulating games to allow the agent to learn optimal strategies.
e. Testing the model: Evaluating the agent's performance against a random player or another agent.

# Building a Tic-Tac-Toe Game with Reinforcement Learning

## a. Setting up the Environment

In [ ]:
import numpy as np
import random

# Define game constants
PLAYER_X = 1
PLAYER_O = -1
EMPTY = 0

# Board dimensions
BOARD_SIZE = 3

## b. Defining the Tic-Tac-Toe Game

In [ ]:
class TicTacToe:
    def __init__(self):
        self.board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)
        self.current_player = PLAYER_X
        self.winner = None
        self.game_over = False

    def reset(self):
        self.board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)
        self.current_player = PLAYER_X
        self.winner = None
        self.game_over = False
        return self.get_state()

    def get_available_moves(self):
        moves = []
        for r in range(BOARD_SIZE):
            for c in range(BOARD_SIZE):
                if self.board[r, c] == EMPTY:
                    moves.append((r, c))
        return moves

    def make_move(self, move):
        if self.game_over or self.board[move[0], move[1]] != EMPTY:
            return False # Invalid move

        self.board[move[0], move[1]] = self.current_player
        self.winner = self._check_winner()
        self.game_over = self.winner is not None or len(self.get_available_moves()) == 0

        if not self.game_over:
            self.current_player *= -1 # Switch player
        return True

    def _check_winner(self):
        # Check rows
        for r in range(BOARD_SIZE):
            if np.all(self.board[r, :] == self.current_player):
                return self.current_player
        # Check columns
        for c in range(BOARD_SIZE):
            if np.all(self.board[:, c] == self.current_player):
                return self.current_player
        # Check diagonals
        if np.all(np.diag(self.board) == self.current_player) or \
           np.all(np.diag(np.fliplr(self.board)) == self.current_player):
            return self.current_player
        return None

    def get_state(self):
        # Convert the board to a tuple to be hashable for Q-table keys
        return tuple(self.board.flatten())

    def display_board(self):
        symbols = {PLAYER_X: 'X', PLAYER_O: 'O', EMPTY: '-'}
        for r in range(BOARD_SIZE):
            row_str = " | ".join([symbols[cell] for cell in self.board[r]])
            print(row_str)
            if r < BOARD_SIZE - 1:
                print("-----------")

## c. Building the Reinforcement Learning Model (Q-Learning Agent)

In [ ]:
class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.q_table = {}
        self.alpha = alpha # Learning rate
        self.gamma = gamma # Discount factor
        self.epsilon = epsilon # Exploration-exploitation trade-off

    def get_q_value(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state, available_moves):
        if not available_moves:
            return None

        if random.uniform(0, 1) < self.epsilon: # Explore
            return random.choice(available_moves)
        else: # Exploit
            q_values = [self.get_q_value(state, move) for move in available_moves]
            max_q = max(q_values)
            # Choose randomly among actions with the maximum Q-value to break ties
            best_moves = [available_moves[i] for i, q in enumerate(q_values) if q == max_q]
            return random.choice(best_moves)

    def learn(self, state, action, reward, next_state, next_available_moves):
        old_q_value = self.get_q_value(state, action)

        if next_state is None: # Terminal state
            next_max_q = 0.0
        else:
            next_q_values = [self.get_q_value(next_state, move) for move in next_available_moves]
            next_max_q = max(next_q_values) if next_q_values else 0.0

        new_q_value = old_q_value + self.alpha * (reward + self.gamma * next_max_q - old_q_value)
        self.q_table[(state, action)] = new_q_value

    def save_q_table(self, filename="q_table.npy"):
        np.save(filename, self.q_table)
        print(f"Q-table saved to {filename}")

    def load_q_table(self, filename="q_table.npy"):
        try:
            self.q_table = np.load(filename, allow_pickle=True).item()
            print(f"Q-table loaded from {filename}")
        except FileNotFoundError:
            print(f"No Q-table found at {filename}. Starting with an empty Q-table.")
        except Exception as e:
            print(f"Error loading Q-table: {e}. Starting with an empty Q-table.")

## d. Training the Model

In [ ]:
def train_agent(agent, num_episodes=10000):
    game = TicTacToe()
    player_o_agent = QLearningAgent(epsilon=0.0) # Player O can be a fixed policy or another learning agent
    # Here, setting epsilon to 0.0 makes it deterministic (exploitation only)
    # but for simplicity for now, it's just a placeholder.

    print(f"Training agent for {num_episodes} episodes...")

    for episode in range(num_episodes):
        state = game.reset()
        done = False

        while not done:
            current_player = game.current_player
            available_moves = game.get_available_moves()

            if current_player == PLAYER_X:
                # Agent (Player X) chooses action
                action = agent.choose_action(state, available_moves)
            else:
                # Opponent (Player O) can be random or another agent
                # For now, let's make Player O choose a random move for training simplicity
                action = random.choice(available_moves)

            # Store current state and action for learning
            old_state = state
            old_action = action

            # Make the move
            game.make_move(action)

            # Get next state and check if game is over
            next_state = game.get_state()
            done = game.game_over

            reward = 0
            if done:
                if game.winner == PLAYER_X:
                    reward = 1 # Player X wins
                elif game.winner == PLAYER_O:
                    reward = -1 # Player O wins (Player X loses)
                else:
                    reward = 0.1 # Draw (small positive reward for not losing)

            # Agent learns from the experience (only Player X is learning here)
            if current_player == PLAYER_X:
                agent.learn(old_state, old_action, reward, next_state, game.get_available_moves())

            state = next_state

        if (episode + 1) % (num_episodes // 10) == 0:
            print(f"Episode {episode + 1}/{num_episodes} completed.")

    print("Training complete.")

# Initialize agent and start training
# You can adjust alpha, gamma, epsilon here
learning_agent = QLearningAgent(alpha=0.2, gamma=0.9, epsilon=0.1)
train_agent(learning_agent, num_episodes=50000)

# Optionally save the Q-table
learning_agent.save_q_table("tic_tac_toe_q_table.npy")

## e. Testing the Model

In [ ]:
def play_game_with_agent(agent, human_player=PLAYER_O):
    game = TicTacToe()

    # Load the Q-table if it was saved from training
    agent.load_q_table("tic_tac_toe_q_table.npy")
    # Set epsilon to 0 for exploitation during testing
    agent.epsilon = 0

    print("\n--- Starting New Game ---")
    game.display_board()

    while not game.game_over:
        current_player = game.current_player
        available_moves = game.get_available_moves()

        if current_player == human_player:
            # Human player's turn
            while True:
                try:
                    row = int(input(f"Player {'X' if human_player == PLAYER_X else 'O'}, enter row (0-2): "))
                    col = int(input(f"Player {'X' if human_player == PLAYER_X else 'O'}, enter column (0-2): "))
                    move = (row, col)
                    if move in available_moves:
                        if game.make_move(move):
                            break
                        else:
                            print("Invalid move. Try again.")
                    else:
                        print("Invalid move. Cell not empty or out of bounds. Try again.")
                except ValueError:
                    print("Invalid input. Please enter numbers for row and column.")
        else:
            # Agent's turn
            print(f"Agent ({'X' if current_player == PLAYER_X else 'O'}) is making a move...")
            state = game.get_state()
            action = agent.choose_action(state, available_moves)
            if action:
                game.make_move(action)
            else:
                # This case should ideally not happen if available_moves is checked
                print("Agent could not find a move. This should not occur.")
                break

        print("\n")
        game.display_board()

    if game.winner == PLAYER_X:
        print("Player X wins!")
    elif game.winner == PLAYER_O:
        print("Player O wins!")
    else:
        print("It's a draw!")
    print("--- Game Over ---\n")

# To play against the agent (agent is Player X, you are Player O)
# Create a new agent instance for playing, it will load the trained q-table
playing_agent = QLearningAgent()
play_game_with_agent(playing_agent, human_player=PLAYER_O)

# If you want to play as Player X against the agent (agent would be Player O)
# You would need to modify the play_game_with_agent function slightly
# to allow the agent to play as PLAYER_O and choose the human_player accordingly.
# For now, the agent plays as PLAYER_X.